In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    roc_auc_score,

    confusion_matrix,

    classification_report
)

from torch.utils.data import Dataset

from transformers import (

    VisualBertModel,

    BertTokenizerFast,

    TrainingArguments,

    Trainer
)

In [ ]:
# ============================================================
# FORCE CPU
# ============================================================

DEVICE = torch.device("cpu")

print(DEVICE)

In [ ]:
# ============================================================
# SET RANDOM SEED
# ============================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

In [ ]:
# ============================================================
# DEFINE PATHS
# ============================================================

DATASET_DIR = "./MultiOFF"

TRAIN_FILE = os.path.join(

    DATASET_DIR,

    "Split Dataset",

    "Training_meme_dataset.csv"
)

DEV_FILE = os.path.join(

    DATASET_DIR,

    "Split Dataset",

    "Validation_meme_dataset.csv"
)

TEST_FILE = os.path.join(

    DATASET_DIR,

    "Split Dataset",

    "Testing_meme_dataset.csv"
)

IMAGE_DIR = os.path.join(

    DATASET_DIR,

    "Labelled Images"
)

print(TRAIN_FILE)

print(DEV_FILE)

print(TEST_FILE)

print(IMAGE_DIR)

In [ ]:
# ============================================================
# LOAD CSV FILES
# ============================================================

train_df = pd.read_csv(TRAIN_FILE)

dev_df = pd.read_csv(DEV_FILE)

test_df = pd.read_csv(TEST_FILE)

print("TRAIN:", len(train_df))

print("DEV:", len(dev_df))

print("TEST:", len(test_df))

In [ ]:
# ============================================================
# CHECK COLUMNS
# ============================================================

print(train_df.columns)

In [ ]:
# ============================================================
# CONVERT LABELS
# ============================================================

label_map = {

    "Non-offensiv": 0,

    "offensive": 1
}

train_df["label"] = train_df["label"].map(label_map)

dev_df["label"] = dev_df["label"].map(label_map)

test_df["label"] = test_df["label"].map(label_map)

In [ ]:
# ============================================================
# REMOVE MISSING IMAGES
# ============================================================

def filter_existing_images(df):

    valid_rows = []

    for _, row in df.iterrows():

        image_path = os.path.join(

            IMAGE_DIR,

            row["image_name"]
        )

        if os.path.exists(image_path):

            valid_rows.append(row)

    return pd.DataFrame(valid_rows)


train_df = filter_existing_images(train_df)

dev_df = filter_existing_images(dev_df)

test_df = filter_existing_images(test_df)

In [ ]:
# ============================================================
# RESET INDICES
# ============================================================

train_df = train_df.reset_index(drop=True)

dev_df = dev_df.reset_index(drop=True)

test_df = test_df.reset_index(drop=True)

In [ ]:
# ============================================================
# LOAD TOKENIZER
# ============================================================

MODEL_NAME = "uclanlp/visualbert-vqa-coco-pre"

tokenizer = BertTokenizerFast.from_pretrained(

    "bert-base-uncased"
)

In [ ]:
# ============================================================
# CREATE DATASET CLASS
# ============================================================

class MultiOFFDataset(Dataset):

    def __init__(

        self,

        dataframe,

        tokenizer,

        image_dir,

        max_length=128
    ):

        self.df = dataframe

        self.tokenizer = tokenizer

        self.image_dir = image_dir

        self.max_length = max_length

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        sample = self.df.iloc[idx]

        text = sample["sentence"]

        label = sample["label"]

        image_path = os.path.join(

            self.image_dir,

            sample["image_name"]
        )

        image = Image.open(

            image_path

        ).convert("RGB")

        image = image.resize((224, 224))

        image = np.array(image).astype(np.float32)

        image = image / 255.0

        visual_embeds = torch.tensor(

            image

        ).permute(2, 0, 1)

        visual_embeds = visual_embeds.mean(dim=0)

        visual_embeds = visual_embeds.flatten()

        visual_embeds = visual_embeds[:2048]

        if visual_embeds.shape[0] < 2048:

            pad_size = (

                2048 -

                visual_embeds.shape[0]
            )

            visual_embeds = torch.cat([

                visual_embeds,

                torch.zeros(pad_size)
            ])

        visual_embeds = visual_embeds.unsqueeze(0)

        visual_attention_mask = torch.ones(

            visual_embeds.shape[:-1],

            dtype=torch.float
        )

        encoding = self.tokenizer(

            text,

            padding="max_length",

            truncation=True,

            max_length=self.max_length,

            return_tensors="pt"
        )

        return {

            "input_ids":

            encoding["input_ids"].squeeze(0),

            "attention_mask":

            encoding["attention_mask"].squeeze(0),

            "visual_embeds":

            visual_embeds,

            "visual_attention_mask":

            visual_attention_mask,

            "labels":

            torch.tensor(

                label,

                dtype=torch.long
            )
        }

In [ ]:
# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = MultiOFFDataset(

    train_df,

    tokenizer,

    IMAGE_DIR
)

val_dataset = MultiOFFDataset(

    dev_df,

    tokenizer,

    IMAGE_DIR
)

test_dataset = MultiOFFDataset(

    test_df,

    tokenizer,

    IMAGE_DIR
)

In [ ]:
# ============================================================
# CREATE VISUALBERT MODEL
# ============================================================

class VisualBERTClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.visualbert = VisualBertModel.from_pretrained(

            MODEL_NAME
        )

        hidden_size = (

            self.visualbert.config.hidden_size
        )

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(

            hidden_size,

            2
        )

    def forward(

        self,

        input_ids,

        attention_mask,

        visual_embeds,

        visual_attention_mask,

        labels=None
    ):

        outputs = self.visualbert(

            input_ids=input_ids,

            attention_mask=attention_mask,

            visual_embeds=visual_embeds,

            visual_attention_mask=

            visual_attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0]

        pooled_output = self.dropout(

            pooled_output
        )

        logits = self.classifier(

            pooled_output
        )

        loss = None

        if labels is not None:

            loss_fn = nn.CrossEntropyLoss()

            loss = loss_fn(

                logits,

                labels
            )

        return {

            "loss": loss,

            "logits": logits
        }

In [ ]:
# ============================================================
# INITIALIZE MODEL
# ============================================================

model = VisualBERTClassifier()

In [ ]:
# ============================================================
# DEFINE METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(

        logits,

        axis=1
    )

    probs = torch.softmax(

        torch.tensor(logits),

        dim=1
    )[:, 1].numpy()

    accuracy = accuracy_score(

        labels,

        preds
    )

    precision = precision_score(

        labels,

        preds
    )

    recall = recall_score(

        labels,

        preds
    )

    f1 = f1_score(

        labels,

        preds
    )

    roc_auc = roc_auc_score(

        labels,

        probs
    )

    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "roc_auc": roc_auc
    }

In [ ]:
# ============================================================
# TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="./visualbert_results",

    num_train_epochs=10,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_steps=10,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    report_to="none"
)

In [ ]:
# ============================================================
# CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
# ============================================================
# TRAIN MODEL
# ============================================================

start_train_time = time.time()

trainer.train()

end_train_time = time.time()

training_time = (

    end_train_time -

    start_train_time
)

print(

    f"Training Time: "

    f"{training_time:.2f} seconds"
)

In [ ]:
# ============================================================
# EVALUATE MODEL
# ============================================================

predictions = trainer.predict(

    test_dataset
)

logits = predictions.predictions

labels = predictions.label_ids

preds = np.argmax(

    logits,

    axis=1
)

probs = torch.softmax(

    torch.tensor(logits),

    dim=1
)[:, 1].numpy()

In [ ]:
# ============================================================
# PERFORMANCE METRICS
# ============================================================

accuracy = accuracy_score(

    labels,

    preds
)

precision = precision_score(

    labels,

    preds
)

recall = recall_score(

    labels,

    preds
)

f1 = f1_score(

    labels,

    preds
)

roc_auc = roc_auc_score(

    labels,

    probs
)

cm = confusion_matrix(

    labels,

    preds
)

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1 Score:", f1)

print("ROC-AUC:", roc_auc)

print()

print(cm)

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(

    classification_report(

        labels,

        preds,

        digits=4
    )
)

In [ ]:
# ============================================================
# INFERENCE TIME
# ============================================================

start_inference = time.time()

_ = trainer.predict(test_dataset)

end_inference = time.time()

total_inference_time = (

    end_inference -

    start_inference
)

average_inference_time = (

    total_inference_time /

    len(test_dataset)
)

print(

    f"Total Inference Time: "

    f"{total_inference_time:.4f} seconds"
)

print(

    f"Average Time Per Sample: "

    f"{average_inference_time:.6f} seconds"
)